# Model Prediksi Time Series: Tutupan Lahan, Emisi, dan Anomali Suhu ASEAN

Notebook ini mengimplementasikan modelling machine learning/deep learning untuk memprediksi `Temperature_Change` pada tahun target `t` menggunakan riwayat tutupan lahan dan emisi dari `t-5` sampai `t-1`.

Pendekatan utama:

- Time series supervised learning dengan sliding window per negara.
- Model utama: LSTM.
- Model pembanding: GRU.
- Evaluasi berbasis waktu agar tidak terjadi data leakage dari masa depan ke masa lalu.
- Simulasi early warning system berdasarkan hasil prediksi anomali suhu.
- Forecast recursive 5 tahun ke depan untuk satu negara.

Dataset yang digunakan:

`Merged_Climate_LandCover_Data_ASEAN.csv`

## 1. Import Library

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)

## 2. Load Dataset

In [ ]:
file_path = "Merged_Climate_LandCover_Data_ASEAN.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"File tidak ditemukan: {file_path}")

df_raw = pd.read_csv(file_path)

print("Jumlah baris dan kolom:", df_raw.shape)
print("Kolom dataset:")
display(pd.DataFrame({"column": df_raw.columns}))

if "Year" not in df_raw.columns or "Area" not in df_raw.columns:
    raise ValueError("Dataset harus memiliki kolom Area dan Year.")

df_raw["Year"] = df_raw["Year"].astype(int)

target_candidates = ["Temperature_Change", "Temperature Change"]
TARGET_COL = next((col for col in target_candidates if col in df_raw.columns), None)
if TARGET_COL is None:
    raise ValueError("Target tidak ditemukan. Gunakan kolom Temperature_Change atau Temperature Change.")

print("Target prediksi:", TARGET_COL)
print("Rentang tahun:", df_raw["Year"].min(), "-", df_raw["Year"].max())
print("Jumlah negara unik:", df_raw["Area"].nunique())

display(df_raw.head())

In [ ]:
missing_info = (
    df_raw.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df_raw))
    .sort_values("missing_pct", ascending=False)
)

display(missing_info.head(20))

## 3. Preprocessing Data

In [ ]:
df = df_raw.copy()
df = df.sort_values(["Area", "Year"]).reset_index(drop=True)
df["Year"] = df["Year"].astype(int)

land_cover_features_requested = [
    "Tree-covered areas",
    "Herbaceous crops",
    "Grassland",
    "Shrub-covered areas",
    "Artificial surfaces (including urban and associated areas)",
    "Mangroves",
    "Inland water bodies",
    "Woody crops",
]

emission_features_requested = [
    "AFOLU",
    "Agrifood systems",
    "Energy",
    "LULUCF",
    "Forest fires",
    "Net Forest conversion",
    "Land-use change",
    "Enteric Fermentation",
]

land_cover_cols = [col for col in land_cover_features_requested if col in df.columns]
emission_cols = [col for col in emission_features_requested if col in df.columns]

candidate_feature_cols = land_cover_cols + emission_cols
print("Fitur tutupan lahan yang tersedia:", land_cover_cols)
print("Fitur emisi yang tersedia:", emission_cols)
print("Jumlah fitur kandidat:", len(candidate_feature_cols))

missing_rate = df[candidate_feature_cols].isna().mean().sort_values(ascending=False)
high_missing_threshold = 0.95
high_missing_cols = missing_rate[missing_rate > high_missing_threshold].index.tolist()

if high_missing_cols:
    print("Fitur dihapus karena missing value terlalu tinggi:", high_missing_cols)

feature_cols_base = [col for col in candidate_feature_cols if col not in high_missing_cols]

# Interpolasi dilakukan per negara agar pola time series tidak tercampur antarnegara.
cols_to_impute = feature_cols_base + [TARGET_COL]
for col in cols_to_impute:
    df[col] = df.groupby("Area")[col].transform(lambda s: s.interpolate(limit_direction="both"))
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

remaining_missing = df[cols_to_impute].isna().sum().sum()
print("Sisa missing value pada fitur dan target:", remaining_missing)

display(df[["Area", "Year", TARGET_COL] + feature_cols_base].head())

In [ ]:
# Feature tambahan untuk menangkap perubahan tahunan tutupan lahan dan emisi.
for col in feature_cols_base:
    df[f"delta_{col}"] = df.groupby("Area")[col].diff().fillna(0)

derived_cols = [col for col in df.columns if col.startswith("delta_")]

feature_cols = feature_cols_base + derived_cols
feature_cols = [col for col in feature_cols if col in df.columns and col != TARGET_COL]

print("Jumlah fitur final:", len(feature_cols))
display(pd.DataFrame({"feature": feature_cols}))

## 4. Feature Engineering Time Series dengan Sliding Window

In [ ]:
LOOKBACK = 5

def create_sequences_by_country(data, feature_cols, target_col, lookback):
    # Membuat sequence time series per negara.
    # Input untuk target tahun t adalah fitur dari tahun t-lookback sampai t-1.
    # Data antar negara tidak dicampur dalam satu sequence.
    X, y, metadata = [], [], []

    for area, group in data.sort_values(["Area", "Year"]).groupby("Area"):
        group = group.sort_values("Year").reset_index(drop=True)
        feature_values = group[feature_cols].to_numpy(dtype="float32")
        target_values = group[target_col].to_numpy(dtype="float32")
        years = group["Year"].to_numpy()

        for i in range(lookback, len(group)):
            X.append(feature_values[i - lookback:i])
            y.append(target_values[i])
            metadata.append({
                "Area": area,
                "target_year": int(years[i]),
                "window_start": int(years[i - lookback]),
                "window_end": int(years[i - 1]),
            })

    return np.array(X, dtype="float32"), np.array(y, dtype="float32"), pd.DataFrame(metadata)


X, y, metadata = create_sequences_by_country(df, feature_cols, TARGET_COL, LOOKBACK)

print("X shape:", X.shape)
print("y shape:", y.shape)
display(metadata.head())

## 5. Train, Validation, dan Test Split Berbasis Waktu

In [ ]:
train_mask = metadata["target_year"] <= 2016
val_mask = (metadata["target_year"] >= 2017) & (metadata["target_year"] <= 2019)
test_mask = metadata["target_year"] >= 2020

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

metadata_train = metadata[train_mask].reset_index(drop=True)
metadata_val = metadata[val_mask].reset_index(drop=True)
metadata_test = metadata[test_mask].reset_index(drop=True)

print("Train:", X_train.shape, y_train.shape, metadata_train["target_year"].min(), "-", metadata_train["target_year"].max())
print("Validation:", X_val.shape, y_val.shape, metadata_val["target_year"].min(), "-", metadata_val["target_year"].max())
print("Test:", X_test.shape, y_test.shape, metadata_test["target_year"].min(), "-", metadata_test["target_year"].max())

if np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any():
    raise ValueError("Masih ada NaN pada fitur sequence.")
if np.isnan(y_train).any() or np.isnan(y_val).any() or np.isnan(y_test).any():
    raise ValueError("Masih ada NaN pada target.")

In [ ]:
# StandardScaler hanya di-fit pada training set untuk menghindari data leakage.
n_features = X_train.shape[-1]
x_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
X_val_scaled = x_scaler.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape)
X_test_scaled = x_scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

print("Normalisasi fitur selesai.")

## 6. Membangun Model LSTM

In [ ]:
def build_lstm_model(lookback, n_features):
    model = Sequential([
        Input(shape=(lookback, n_features)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"],
    )
    return model


lstm_model = build_lstm_model(LOOKBACK, n_features)
lstm_model.summary()

## 7. Training Model LSTM

In [ ]:
EPOCHS = 100
BATCH_SIZE = 16

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=7,
        min_lr=1e-5,
    ),
]

lstm_history = lstm_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
def plot_training_history(history, title):
    hist = pd.DataFrame(history.history)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.lineplot(data=hist[["loss", "val_loss"]], ax=axes[0])
    axes[0].set_title(f"{title}: Training Loss vs Validation Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("MSE Loss")

    sns.lineplot(data=hist[["mae", "val_mae"]], ax=axes[1])
    axes[1].set_title(f"{title}: Training MAE vs Validation MAE")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MAE")

    plt.tight_layout()
    plt.show()

    final_train_loss = hist["loss"].iloc[-1]
    final_val_loss = hist["val_loss"].iloc[-1]
    print("Interpretasi singkat:")
    if final_val_loss > final_train_loss * 1.5:
        print("- Validation loss jauh lebih tinggi daripada training loss. Ada indikasi overfitting.")
    elif final_train_loss > 0.20 and final_val_loss > 0.20:
        print("- Training dan validation loss sama-sama relatif tinggi. Ada indikasi underfitting.")
    else:
        print("- Training dan validation loss cukup seimbang. Model relatif stabil pada validation set.")


plot_training_history(lstm_history, "LSTM")

## 8. Evaluasi Model LSTM

In [ ]:
def regression_metrics(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


lstm_val_pred = lstm_model.predict(X_val_scaled, verbose=0).ravel()
lstm_test_pred = lstm_model.predict(X_test_scaled, verbose=0).ravel()

lstm_metrics = regression_metrics(y_test, lstm_test_pred, "LSTM")
results_df = pd.DataFrame([
    regression_metrics(y_val, lstm_val_pred, "LSTM Validation"),
    lstm_metrics,
])

display(results_df)

In [ ]:
lstm_pred_df = metadata_test.copy()
lstm_pred_df["Actual Temperature Change"] = y_test
lstm_pred_df["Predicted Temperature Change"] = lstm_test_pred
lstm_pred_df["Error"] = lstm_pred_df["Predicted Temperature Change"] - lstm_pred_df["Actual Temperature Change"]

display(lstm_pred_df.head(20))

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(
    data=lstm_pred_df,
    x="Actual Temperature Change",
    y="Predicted Temperature Change",
    hue="target_year",
    palette="viridis",
)
min_val = min(lstm_pred_df["Actual Temperature Change"].min(), lstm_pred_df["Predicted Temperature Change"].min())
max_val = max(lstm_pred_df["Actual Temperature Change"].max(), lstm_pred_df["Predicted Temperature Change"].max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--")
plt.title("LSTM: Actual vs Predicted Temperature Change")
plt.tight_layout()
plt.show()

In [ ]:
requested_countries = ["Indonesia", "Malaysia", "Philippines", "Nigeria", "Colombia"]
available_countries = [c for c in requested_countries if c in lstm_pred_df["Area"].unique()]

if len(available_countries) < 3:
    additional = [c for c in lstm_pred_df["Area"].unique() if c not in available_countries]
    available_countries.extend(additional[: 3 - len(available_countries)])

print("Negara yang diplot:", available_countries)
print("Catatan: Nigeria dan Colombia tidak ada di file ASEAN, sehingga hanya diplot jika tersedia di dataset.")

for country in available_countries:
    country_df = lstm_pred_df[lstm_pred_df["Area"] == country].sort_values("target_year")
    plt.figure(figsize=(9, 4))
    plt.plot(country_df["target_year"], country_df["Actual Temperature Change"], marker="o", label="Aktual")
    plt.plot(country_df["target_year"], country_df["Predicted Temperature Change"], marker="o", label="Prediksi LSTM")
    plt.title(f"LSTM: Aktual vs Prediksi - {country}")
    plt.xlabel("Tahun Target")
    plt.ylabel("Temperature Change")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9. Perbandingan dengan GRU

In [ ]:
def build_gru_model(lookback, n_features):
    model = Sequential([
        Input(shape=(lookback, n_features)),
        GRU(64, return_sequences=True),
        Dropout(0.2),
        GRU(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1),
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"],
    )
    return model


gru_model = build_gru_model(LOOKBACK, n_features)
gru_model.summary()

In [ ]:
gru_history = gru_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

plot_training_history(gru_history, "GRU")

In [ ]:
gru_val_pred = gru_model.predict(X_val_scaled, verbose=0).ravel()
gru_test_pred = gru_model.predict(X_test_scaled, verbose=0).ravel()

comparison_df = pd.DataFrame([
    regression_metrics(y_test, lstm_test_pred, "LSTM"),
    regression_metrics(y_test, gru_test_pred, "GRU"),
])

display(comparison_df)

best_by_rmse = comparison_df.sort_values(["RMSE", "R2"], ascending=[True, False]).iloc[0]
print(f"Model terbaik berdasarkan RMSE terkecil: {best_by_rmse['Model']} (RMSE={best_by_rmse['RMSE']:.4f}, R2={best_by_rmse['R2']:.4f})")

In [ ]:
best_model_name = best_by_rmse["Model"]
best_model = lstm_model if best_model_name == "LSTM" else gru_model
best_test_pred = lstm_test_pred if best_model_name == "LSTM" else gru_test_pred

best_pred_df = metadata_test.copy()
best_pred_df["Actual Temperature Change"] = y_test
best_pred_df["Predicted Temperature Change"] = best_test_pred
best_pred_df["Error"] = best_pred_df["Predicted Temperature Change"] - best_pred_df["Actual Temperature Change"]

display(best_pred_df.head())

## 10. Simulasi Early Warning System

In [ ]:
def classify_warning(predicted_temp):
    if predicted_temp < 1.0:
        return "Aman"
    elif predicted_temp < 1.5:
        return "Waspada"
    elif predicted_temp < 2.0:
        return "Bahaya"
    else:
        return "Kritis"


warning_df = best_pred_df.copy()
warning_df["Warning Level"] = warning_df["Predicted Temperature Change"].apply(classify_warning)

display(warning_df.sort_values(["target_year", "Area"]).head(30))

In [ ]:
warning_order = ["Aman", "Waspada", "Bahaya", "Kritis"]

plt.figure(figsize=(8, 4))
sns.countplot(
    data=warning_df,
    x="Warning Level",
    order=warning_order,
    palette="viridis",
)
plt.title("Jumlah Sampel Negara-Tahun pada Tiap Level Early Warning")
plt.xlabel("Warning Level")
plt.ylabel("Jumlah Sampel")
plt.tight_layout()
plt.show()

display(
    warning_df.groupby(["target_year", "Warning Level"])
    .size()
    .reset_index(name="count")
    .sort_values(["target_year", "Warning Level"])
)

## 11. Forecast 5 Tahun ke Depan

In [ ]:
def forecast_next_year_features(history_df, feature_cols, method="trend_3y"):
    # Membuat estimasi fitur tahun berikutnya.
    # method="last": semua fitur mengikuti nilai tahun terakhir.
    # method="trend_3y": fitur tahun berikutnya mengikuti rata-rata perubahan 3 tahun terakhir.
    next_row = history_df.iloc[-1].copy()
    next_row["Year"] = int(history_df["Year"].iloc[-1] + 1)

    if method == "last" or len(history_df) < 4:
        for col in feature_cols_base:
            next_row[col] = history_df[col].iloc[-1]
        return next_row

    recent = history_df.tail(4)
    for col in feature_cols_base:
        avg_delta = recent[col].diff().dropna().mean()
        if pd.isna(avg_delta):
            avg_delta = 0.0
        next_row[col] = history_df[col].iloc[-1] + avg_delta

    return next_row


def recompute_delta_features(country_history):
    country_history = country_history.sort_values("Year").copy()
    for col in feature_cols_base:
        delta_col = f"delta_{col}"
        if delta_col in country_history.columns:
            country_history[delta_col] = country_history[col].diff().fillna(0)
    return country_history


def forecast_country_5_years(country, model, model_name, years_ahead=5, feature_method="trend_3y"):
    if country not in df["Area"].unique():
        raise ValueError(f"Negara tidak tersedia di dataset: {country}")

    country_history = df[df["Area"] == country].sort_values("Year").copy()
    forecast_rows = []

    for _ in range(years_ahead):
        input_window = country_history.tail(LOOKBACK).copy()
        if len(input_window) < LOOKBACK:
            raise ValueError(f"Riwayat {country} kurang dari {LOOKBACK} tahun.")

        X_future = input_window[feature_cols].to_numpy(dtype="float32")[None, :, :]
        X_future_scaled = x_scaler.transform(X_future.reshape(-1, n_features)).reshape(X_future.shape)
        predicted_temp = float(model.predict(X_future_scaled, verbose=0).ravel()[0])

        next_row = forecast_next_year_features(country_history, feature_cols_base, method=feature_method)
        next_row[TARGET_COL] = predicted_temp
        next_row["Area"] = country
        country_history = pd.concat([country_history, pd.DataFrame([next_row])], ignore_index=True)
        country_history = recompute_delta_features(country_history)

        forecast_rows.append({
            "Area": country,
            "Year": int(next_row["Year"]),
            "Model": model_name,
            "Predicted Temperature Change": predicted_temp,
            "Warning Level": classify_warning(predicted_temp),
        })

    return pd.DataFrame(forecast_rows)


forecast_country = "Indonesia" if "Indonesia" in df["Area"].unique() else df["Area"].iloc[0]
forecast_df = forecast_country_5_years(
    country=forecast_country,
    model=best_model,
    model_name=best_model_name,
    years_ahead=5,
    feature_method="trend_3y",
)

display(forecast_df)

In [ ]:
plt.figure(figsize=(9, 4))
sns.lineplot(
    data=forecast_df,
    x="Year",
    y="Predicted Temperature Change",
    marker="o",
)
for _, row in forecast_df.iterrows():
    plt.text(row["Year"], row["Predicted Temperature Change"], row["Warning Level"], ha="center", va="bottom")
plt.title(f"Forecast 5 Tahun ke Depan - {forecast_country} ({best_model_name})")
plt.xlabel("Tahun")
plt.ylabel("Predicted Temperature Change")
plt.tight_layout()
plt.show()

## 12. Interpretasi Akhir

**Mengapa LSTM/GRU cocok?**  
LSTM dan GRU cocok untuk data ini karena target anomali suhu pada tahun tertentu tidak hanya dipengaruhi oleh kondisi pada tahun yang sama, tetapi juga oleh riwayat perubahan tutupan lahan dan emisi pada beberapa tahun sebelumnya. Dengan input sequence `t-5` sampai `t-1`, model recurrent dapat membaca pola temporal dan efek jeda waktu atau lagged effect.

**Bagaimana hubungan temporal ditangkap?**  
Setiap sampel model berisi 5 tahun historis fitur tutupan lahan dan emisi untuk satu negara. Sequence dibuat per negara sehingga urutan waktu Indonesia, Malaysia, atau negara lain tidak tercampur. LSTM/GRU kemudian mempelajari pola perubahan fitur antar tahun, termasuk fitur turunan `delta_`, untuk memprediksi `Temperature_Change` pada tahun target.

**Makna metrik evaluasi.**  
MAE menunjukkan rata-rata besar kesalahan absolut prediksi dalam satuan derajat Celsius. MSE dan RMSE memberi penalti lebih besar pada error yang tinggi; RMSE lebih mudah dibaca karena kembali ke satuan target. R2 menunjukkan seberapa besar variasi target yang dapat dijelaskan model pada test set. RMSE lebih kecil dan R2 lebih besar menunjukkan performa yang lebih baik.

**Early warning system.**  
Prediksi `Temperature_Change` dikategorikan menjadi `Aman`, `Waspada`, `Bahaya`, dan `Kritis`. Kategori ini dapat membantu membaca hasil model sebagai sinyal risiko negara-tahun, bukan hanya angka prediksi mentah.

**Keterbatasan model.**

- Data hanya mencakup 1992-2022, sehingga jumlah titik waktu per negara relatif pendek untuk deep learning.
- Anomali suhu tidak hanya dipengaruhi oleh tutupan lahan dan emisi lokal, tetapi juga faktor global seperti sirkulasi atmosfer, laut, ENSO, aerosol, dan variabilitas iklim alami.
- Forecast 5 tahun ke depan membutuhkan asumsi fitur masa depan, misalnya tren rata-rata 3 tahun terakhir. Jika asumsi fitur berubah, hasil forecast juga berubah.
- Hubungan korelasi antara fitur dan target tidak selalu berarti kausalitas langsung.
- Dataset ASEAN tidak memuat Nigeria dan Colombia, sehingga visualisasi negara tersebut hanya dapat dilakukan jika dataset diganti ke cakupan tropis/global.

## 13. Output yang Dihasilkan

Notebook ini menghasilkan:

- Data preprocessing bersih dari missing value bermasalah.
- Sequence time series dengan `LOOKBACK = 5`.
- Model LSTM sebagai model utama.
- Model GRU sebagai pembanding.
- Evaluasi MAE, MSE, RMSE, dan R2.
- Grafik training loss dan MAE.
- Grafik actual vs predicted.
- Simulasi early warning system.
- Forecast recursive 5 tahun ke depan untuk satu negara.
- Interpretasi hasil dalam bahasa Indonesia.